# Lab 10: Backtesting and Validation

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab10_backtesting.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Objective

An investment committee sees the best historical Sharpe among 200
related strategies. Decide how much weight to give that result. You will
simulate the candidates, compare their ranks across different parts of
the history, and test the winning Sharpe against a benchmark that
accounts for the search.

You set the true mean return of every candidate to zero in the first
simulation. In the second, you add a known weak signal to ten candidates
and check how the evidence changes.

**Time estimate:** 90–120 minutes

After the lab, you should be able to:

-   Explain why a selected Sharpe is the maximum of a search
-   Separate temporal walk-forward validation from CSCV
-   Interpret PBO as rank instability within a declared candidate garden
-   Compare PSR against zero with DSR against a search-adjusted
    benchmark
-   State the trial-count and dependence assumptions behind DSR
-   Keep statistical search, implementation costs, and investment
    decisions separate

> **Assessment boundary**
>
> In FIN510 Coursework 2, **Option A (factor replication)** uses HAC
> standard errors for its time-series regressions. **Option B (market
> prediction)** compares OLS with ridge or lasso using expanding-window
> walk-forward evaluation. The search diagnostics in this lab apply when
> comparing a set of candidate trading strategies. Follow the CW2 brief
> on Blackboard for the required analysis.

## Setup

The chapter and lab use the same calculation functions in
`src/fdslab/validation.py`. The standalone notebook includes a copy of
those functions in its first code cell. The local website build imports
them from this project.

In [ ]:
# Run this setup cell to load the course validation functions.
# Generated from src/fdslab/validation.py.

import math
from dataclasses import dataclass
from typing import Optional, Sequence, Tuple, Union

import numpy as np
from scipy.stats import norm


def sharpe_ratio(returns: np.ndarray, eps: float = 1e-12) -> float:
    """Compute the sample Sharpe ratio for a 1D array."""
    x = np.asarray(returns, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    return float(mu / (sd + eps))


def probabilistic_sharpe_ratio(
    sr_hat: float,
    sr_benchmark: float,
    n_obs: int,
    skew: float = 0.0,
    kurtosis: float = 3.0,
) -> float:
    """Probabilistic Sharpe Ratio (PSR)."""
    if n_obs <= 1:
        return float("nan")
    num = (sr_hat - sr_benchmark) * math.sqrt(max(n_obs - 1, 1))
    den = math.sqrt(
        max(1.0 - skew * sr_hat + ((kurtosis - 1.0) / 4.0) * sr_hat * sr_hat, 1e-12)
    )
    z = num / den
    return float(norm.cdf(z))


@dataclass
class CSCVResult:
    pbo: float
    taus: np.ndarray
    oos_ranks: np.ndarray
    splits_used: int
    sr_is: np.ndarray
    sr_oos: np.ndarray


def _contiguous_folds(T: int, k: int) -> Sequence[np.ndarray]:
    idx = np.arange(T)
    sizes = np.full(k, T // k)
    sizes[: T % k] += 1
    folds = []
    start = 0
    for s in sizes:
        folds.append(idx[start : start + s])
        start += s
    return folds


def _choose_half(k: int) -> np.ndarray:
    from itertools import combinations

    half = k // 2
    combs = list(combinations(range(k), half))
    masks = np.zeros((len(combs), k), dtype=bool)
    for i, comb in enumerate(combs):
        masks[i, list(comb)] = True
    return masks


def cscv_pbo(
    strategies: np.ndarray,
    n_folds: int = 10,
    max_splits: Optional[int] = None,
) -> CSCVResult:
    """Estimate Probability of Backtest Overfitting (PBO) via CSCV."""
    X = np.asarray(strategies, dtype=float)
    if X.ndim != 2:
        raise ValueError("strategies must be 2D (T, N)")
    T, N = X.shape
    if n_folds < 2 or n_folds > T:
        raise ValueError("n_folds must be >=2 and <= T")
    if n_folds % 2 != 0:
        n_folds += 1
        n_folds = min(n_folds, T)

    folds = _contiguous_folds(T, n_folds)
    masks = _choose_half(n_folds)
    if max_splits is not None and max_splits < len(masks):
        rng = np.random.default_rng(42)
        sel = rng.choice(len(masks), size=max_splits, replace=False)
        masks = masks[sel]

    taus = []
    oos_ranks = []
    sr_is = []
    sr_oos_star = []

    for m in masks:
        ins_idx = np.concatenate([folds[i] for i in range(n_folds) if m[i]])
        oos_idx = np.concatenate([folds[i] for i in range(n_folds) if not m[i]])

        sr_ins = np.array([sharpe_ratio(X[ins_idx, j]) for j in range(N)])
        j_star = int(np.nanargmax(sr_ins))

        sr_oos = np.array([sharpe_ratio(X[oos_idx, j]) for j in range(N)])
        ranks = np.argsort(-sr_oos)
        rank_pos = int(np.where(ranks == j_star)[0][0]) + 1
        oos_ranks.append(rank_pos)
        sr_is.append(sr_ins[j_star])
        sr_oos_star.append(sr_oos[j_star])

        omega = 1.0 - rank_pos / (N + 1.0)
        tau = math.log(omega / (1.0 - omega))
        taus.append(tau)

    taus = np.array(taus, dtype=float)
    oos_ranks = np.array(oos_ranks, dtype=int)
    pbo = float(np.mean(taus < 0.0))
    return CSCVResult(
        pbo=pbo,
        taus=taus,
        oos_ranks=oos_ranks,
        splits_used=len(taus),
        sr_is=np.array(sr_is, dtype=float),
        sr_oos=np.array(sr_oos_star, dtype=float),
    )


def deflated_sharpe_ratio(
    sr_hat: float,
    sr_trials: Sequence[float],
    n_obs: int,
    skew: float = 0.0,
    kurtosis: float = 3.0,
    rho: float = 0.0,
) -> Tuple[float, float]:
    """Deflated Sharpe Ratio (DSR) per Bailey and López de Prado (2014).

    ``rho`` is the estimated average correlation among the trial return
    series. Appendix A.3 maps ``M`` dependent trials to the implied number
    of independent trials ``rho + (1 - rho) * M``. The dispersion of the
    observed Sharpe ratios remains the sample dispersion; correlation acts
    through the effective trial count. Between one and two effective trials,
    interpolate linearly from zero to the two-trial Sharpe benchmark.
    """
    trials = np.asarray(list(sr_trials), dtype=float)
    n_trials = int(trials.size)
    if n_trials < 2:
        raise ValueError("sr_trials must contain at least 2 Sharpe ratios")
    min_rho = -1.0 / (n_trials - 1.0)
    if not min_rho <= rho <= 1.0:
        raise ValueError(
            f"rho must be between {min_rho:.6f} and 1 for {n_trials} trials"
        )

    sigma_sr = float(np.std(trials, ddof=1))
    n_effective = float(rho + (1.0 - rho) * n_trials)
    sr_star = float(expected_max_sharpe(max(n_effective, 2.0), sigma_sr))
    if n_effective < 2.0:
        sr_star *= n_effective - 1.0
    dsr = probabilistic_sharpe_ratio(sr_hat, sr_star, n_obs, skew=skew, kurtosis=kurtosis)
    return float(dsr), float(sr_star)


def expected_max_sharpe(n_trials: Union[int, np.ndarray], sigma_sr: float) -> np.ndarray:
    """Selection-adjusted Sharpe SR* under a zero-skill null.

    Bailey and López de Prado (2014); AFML §14.7.3. The expected maximum
    of ``n_trials`` independent Sharpe estimates with cross-trial standard
    deviation ``sigma_sr``.
    """
    n = np.asarray(n_trials, dtype=float)
    if np.any(n < 2):
        raise ValueError("n_trials must be at least 2")
    gamma = 0.5772156649015329
    return sigma_sr * (
        (1.0 - gamma) * norm.ppf(1.0 - 1.0 / n)
        + gamma * norm.ppf(1.0 - 1.0 / (n * math.e))
    )


def generate_noise_strategies(T: int, N: int, rho: float = 0.0, seed: int = 7) -> np.ndarray:
    """Simulate N strategies with no true edge."""
    rng = np.random.default_rng(seed)
    if rho < 1e-12:
        return rng.normal(0.0, 1.0, size=(T, N))
    z = rng.normal(0.0, 1.0, size=(T, 1))
    e = rng.normal(0.0, 1.0, size=(T, N))
    x = math.sqrt(rho) * z + math.sqrt(max(1.0 - rho, 0.0)) * e
    return x


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if "cscv_pbo" not in globals():
    for project in (Path.cwd(), Path.cwd().parent):
        utilities = project / "scripts" / "utilities"
        if (utilities / "overfit_metrics.py").is_file():
            sys.path.insert(0, str(utilities))
            break
    from overfit_metrics import (
        cscv_pbo,
        deflated_sharpe_ratio,
        expected_max_sharpe,
        generate_noise_strategies,
        probabilistic_sharpe_ratio,
        sharpe_ratio,
    )

ANNUALISE = np.sqrt(12)

## Part 1: Name the validation question

Walk-forward and combinatorially symmetric cross-validation (CSCV) use
time blocks for different purposes.

**Walk-forward** asks: if the model had been refitted using information
available at each date, how would the sequence of next-period forecasts
have performed? Its test observations follow a deployment chronology.
Each preprocessing and tuning step belongs inside the training window.

**CSCV** asks: when the best candidate is chosen in one balanced half of
the history, where does that same candidate rank in the complementary
half? It recombines contiguous slices symmetrically. It estimates
selection instability, not a live trading path (Bailey et al. 2015).

Classify each design before writing code:

1.  Fit through December 2018, predict January 2019, expand by one
    month, and repeat.
2.  Divide twenty years into ten slices, choose five as in sample, and
    use the other five out of sample for each symmetric combination.
3.  Shuffle months into five random folds before estimating a next-month
    return model.
4.  Fit on 2000–2018, inspect 2019–2023, revise the lookback, and score
    2019–2023 again.

## Part 2: Build a noise garden

Generate a **240 × 200 return matrix**. Each row is one month and each
column is a strategy. The simulator gives every strategy population mean
zero and shares a common shock across strategies to set their population
pairwise correlation to 0.2. The estimated correlations in these 240
months will vary around that target.

Compute one monthly Sharpe per column as the sample mean divided by the
sample standard deviation. The simulations treat these as excess returns
and generate months independently. For reporting, multiply each monthly
Sharpe by $\sqrt{12}$ to show it on an annualised scale. Select the
largest sample Sharpe.

In [ ]:
T = 240
M = 200
RHO = 0.2
SEED = 42

noise = generate_noise_strategies(T=T, N=M, rho=RHO, seed=SEED)

mean_pairwise_corr = (
    pd.DataFrame(noise).corr().to_numpy()[np.triu_indices(M, k=1)].mean()
)
trial_sharpes = np.array([sharpe_ratio(noise[:, j]) for j in range(M)])
winner = int(np.argmax(trial_sharpes))

garden_audit = pd.Series({
    "Months": T,
    "Strategies": M,
    "Population mean of every strategy": 0.0,
    "Target pairwise correlation": RHO,
    "Realised mean pairwise correlation": mean_pairwise_corr,
    "Winning monthly Sharpe": trial_sharpes[winner],
    "Winning annualised Sharpe": trial_sharpes[winner] * ANNUALISE,
})

print("=== Noise garden audit ===")
print(garden_audit.round(4))

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.hist(trial_sharpes * ANNUALISE, bins=28, color="steelblue", alpha=0.85)
ax.axvline(
    trial_sharpes[winner] * ANNUALISE,
    color="coral",
    linestyle="--",
    linewidth=2,
    label=f"Selected winner: {trial_sharpes[winner] * ANNUALISE:.3f}",
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Annualised sample Sharpe")
ax.set_ylabel("Strategies")
ax.set_title("The winner of 200 zero-edge strategies")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Answer from the output:

1.  Why is the realised mean pairwise correlation close to, but not
    exactly, 0.2?
2.  How can the selected strategy have a positive Sharpe when every
    population mean is zero?
3.  What information would be lost if the researcher showed only the
    coral line?

## Part 3: Price the maximum

The false-strategy benchmark $SR^*$ is the expected maximum Sharpe under
a zero-edge null. It rises with the number of independent trials and
with the dispersion of Sharpe estimates (Bailey and Prado 2014).

Estimate dispersion from the 200 monthly sample Sharpes using their
sample standard deviation. Keep that number fixed while varying the
**hypothetical independent trial count**. The slide’s comparison figure
uses illustrative dispersion values. This table instead uses the
dispersion measured in your simulation. Part 5 accounts for dependence
in the original 200 trials.

In [ ]:
sigma_sr = trial_sharpes.std(ddof=1)
trial_grid = np.array([2, 5, 10, 20, 50, 100, 200, 500, 1000])
sr_star_grid = expected_max_sharpe(trial_grid, sigma_sr)

benchmark_table = pd.DataFrame({
    "Independent trials": trial_grid,
    "Expected maximum monthly Sharpe": sr_star_grid,
    "Expected maximum annualised Sharpe": sr_star_grid * ANNUALISE,
})

print("=== Expected zero-edge winner ===")
print(f"Cross-trial monthly Sharpe standard deviation: {sigma_sr:.4f}")
print(benchmark_table.round(3).to_string(index=False))

1.  Does doubling the number of trials double $SR^*$?
2.  Why does a short or noisy backtest have a larger $SR^*$ than a long,
    stable one with the same number of trials?
3.  If several strategy variants are tried informally and discarded, do
    they belong in the trial count?

## Part 4: Test rank stability

CSCV divides the 240 months into ten contiguous slices. For each
selected combination of five slices, it chooses the highest-Sharpe
strategy in sample and records the same strategy’s Sharpe and rank in
the complementary five slices.

There are $\binom{10}{5}=252$ ways to choose the in-sample slices. We
use a fixed subsample of 150 splits. The helper uses seed 42 to choose
the splits.

In one split, rank the same candidate that won in sample against all
candidates out of sample. With rank $r=1$ best and $N$ candidates, the
helper sets $\omega=1-r/(N+1)$. A champion ranked fourth of five has
$\omega=1-4/6=1/3$, below the median rank. Its rank logit
$\log[\omega/(1-\omega)]$ is negative, so the split counts towards PBO.

In [ ]:
noise_cscv = cscv_pbo(noise, n_folds=10, max_splits=150)

annual_is = noise_cscv.sr_is * ANNUALISE
annual_oos = noise_cscv.sr_oos * ANNUALISE
slope, intercept = np.polyfit(annual_is, annual_oos, 1)
below_median = int(np.count_nonzero(noise_cscv.taus < 0))
negative_sharpe = int(np.count_nonzero(annual_oos < 0))

print("=== CSCV on the noise garden ===")
print(f"Splits used: {noise_cscv.splits_used}")
print(f"PBO: {below_median}/{noise_cscv.splits_used} = {noise_cscv.pbo:.3f}")
print(f"Champion OOS Sharpe below zero: {negative_sharpe}/{noise_cscv.splits_used} = {(annual_oos < 0).mean():.3f}")
print(f"Slope of champion OOS Sharpe on IS Sharpe: {slope:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(annual_is, annual_oos, s=22, alpha=0.7, color="steelblue")
x_line = np.linspace(annual_is.min(), annual_is.max(), 80)
axes[0].plot(x_line, slope * x_line + intercept, color="coral", linewidth=2)
axes[0].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Champion in-sample Sharpe (annualised)")
axes[0].set_ylabel("Same strategy out-of-sample Sharpe (annualised)")
axes[0].set_title("Performance degradation")

axes[1].hist(noise_cscv.taus, bins=20, color="steelblue", alpha=0.85)
axes[1].axvline(0, color="coral", linestyle="--", linewidth=2)
axes[1].set_xlabel("Logit of out-of-sample rank")
axes[1].set_ylabel("CSCV splits")
axes[1].set_title("Rank stability")

plt.tight_layout()
plt.show()

The right panel counts below-median ranks to the left of zero. The left
panel shows the same winners’ Sharpe ratios in each split. Count points
below its zero line to find how many champions had negative
out-of-sample Sharpe. These two counts can differ because the median
Sharpe can sit above or below zero.

1.  Why should exchangeable noise produce PBO near 0.5?
2.  What does the fitted slope say about the most impressive in-sample
    champions in this draw?
3.  Why is this figure not a walk-forward equity curve?
4.  Can PBO tell whether the out-of-sample return is positive after
    costs?

## Part 5: Raise the Sharpe benchmark

The probabilistic Sharpe ratio (PSR) uses a normal approximation to
assess whether the selected Sharpe exceeds a stated benchmark. Its
standard error reflects sample length, skewness and kurtosis. A winner
selected from many zero-edge strategies often clears zero by chance.

The deflated Sharpe ratio (DSR) uses the expected maximum under the
search as the benchmark. For $M$ correlated trials, Appendix A.3 of
Bailey and Prado (2014) approximates the effective number of independent
trials as:

$$
N_{\mathrm{eff}}=\rho+(1-\rho)M.
$$

Here $M=200$ counts the candidates and $\rho=0.2$ is the known
population correlation imposed by the simulator. The approximation gives
$0.2+0.8(200)=160.2$ effective trials. The helper uses the observed
cross-trial dispersion. The two PSR calculations use the same 240
observations and the same **monthly** selected Sharpe. Only the
benchmark changes. We annualise Sharpes for display after the
probability calculations.

In [ ]:
winner_returns = noise[:, winner]
winner_skew = float(pd.Series(winner_returns).skew())
winner_kurtosis = float(pd.Series(winner_returns).kurtosis() + 3.0)
winner_sr = trial_sharpes[winner]

psr_zero = probabilistic_sharpe_ratio(
    sr_hat=winner_sr,
    sr_benchmark=0.0,
    n_obs=T,
    skew=winner_skew,
    kurtosis=winner_kurtosis,
)

dsr_noise, sr_star_noise = deflated_sharpe_ratio(
    sr_hat=winner_sr,
    sr_trials=trial_sharpes,
    n_obs=T,
    skew=winner_skew,
    kurtosis=winner_kurtosis,
    rho=RHO,
)

n_effective = RHO + (1.0 - RHO) * M

selection_table = pd.Series({
    "Selected monthly Sharpe": winner_sr,
    "Selected annualised Sharpe": winner_sr * ANNUALISE,
    "Effective trial count": n_effective,
    "Monthly selection benchmark SR*": sr_star_noise,
    "Annualised selection benchmark SR*": sr_star_noise * ANNUALISE,
    "PSR against zero": psr_zero,
    "DSR against SR*": dsr_noise,
    "PSR minus DSR (before rounding)": psr_zero - dsr_noise,
})

print("=== Selection-adjusted Sharpe evidence ===")
print(selection_table.round(3))

1.  Why can PSR against zero be near one in a garden known to contain no
    edge?
2.  What does the PSR minus DSR gap measure in this design?
3.  Why must the report state $M$, $\rho$, and the Sharpe dispersion
    rather than giving DSR alone?
4.  What limitation remains in the average-correlation approximation?

## Part 6: Add a known edge

Start from the same seed and keep the same simulated monthly shocks.
Each strategy has population monthly standard deviation 1. Add
$0.50/\sqrt{12}\approx0.144$ to each monthly return in the first ten
columns. Their annualised population Sharpe becomes 0.50, while the
other 190 still have population Sharpe zero. The new run shows what
changes when we alter only those ten population means.

In [ ]:
edge = generate_noise_strategies(T=T, N=M, rho=RHO, seed=SEED)
edge[:, :10] += 0.50 / ANNUALISE

edge_sharpes = np.array([sharpe_ratio(edge[:, j]) for j in range(M)])
edge_winner = int(np.argmax(edge_sharpes))
edge_cscv = cscv_pbo(edge, n_folds=10, max_splits=150)

edge_returns = edge[:, edge_winner]
edge_skew = float(pd.Series(edge_returns).skew())
edge_kurtosis = float(pd.Series(edge_returns).kurtosis() + 3.0)
edge_psr = probabilistic_sharpe_ratio(
    edge_sharpes[edge_winner], 0.0, T, edge_skew, edge_kurtosis
)
edge_dsr, edge_sr_star = deflated_sharpe_ratio(
    sr_hat=edge_sharpes[edge_winner],
    sr_trials=edge_sharpes,
    n_obs=T,
    skew=edge_skew,
    kurtosis=edge_kurtosis,
    rho=RHO,
)

comparison = pd.DataFrame({
    "Noise garden": [
        winner_sr * ANNUALISE,
        noise_cscv.pbo,
        psr_zero,
        sr_star_noise * ANNUALISE,
        dsr_noise,
    ],
    "Weak-edge garden": [
        edge_sharpes[edge_winner] * ANNUALISE,
        edge_cscv.pbo,
        edge_psr,
        edge_sr_star * ANNUALISE,
        edge_dsr,
    ],
}, index=[
    "Selected annualised Sharpe",
    "PBO",
    "PSR against zero",
    "Annualised SR*",
    "DSR",
])

print("=== Known-truth comparison ===")
print(comparison.round(3))
print(f"Weak-edge winner is one of the ten edged strategies: {edge_winner < 10}")

1.  Did PBO and DSR move in the expected direction?
2.  Why did PBO stay above zero and DSR below one?
3.  What would happen if the same mean shift were spread across all 200
    strategies?
4.  Why does detecting an edge still not establish an investable
    strategy?

## Part 7: Write the research decision

Write a 150-word memo about the **noise garden**. Use the actual outputs
from your run and include:

-   The candidate count, history length, and correlation assumption
-   The selected annualised Sharpe
-   PBO and what its denominator is
-   PSR against zero, DSR, and the selection-adjusted $SR^*$
-   A decision to reject, continue researching, or paper trade
-   One limitation of the simulation

Then add one sentence explaining what changed in the weak-edge garden
without claiming that the exercise proves those ten strategies would
work live.

## Reporting template

For a real strategy garden, report:

-   **Data:** universe, sample, point-in-time treatment, label horizon,
    and costs
-   **Search:** every trialled configuration, selection metric, and
    failed variants
-   **Chronology:** walk-forward design, window length, purge or embargo
    where labels overlap
-   **CSCV:** number of slices and splits, PBO, logit distribution and
    the share of winners with negative complementary Sharpe
-   **Sharpe evidence:** PSR benchmark, $M$, estimated $\rho$, effective
    $N$, $SR^*$, and DSR
-   **Decision:** thresholds fixed before seeing the outputs and the
    action they imply

A single pre-specified factor regression does not need a manufactured
garden of variants. Report HAC inference and appropriate stability
checks for that estimand. A strategy chosen from many alternatives does
need the search recorded next to its winner.

## References

Bailey, David H., Jonathan M. Borwein, Marcos López de Prado, and Qiji
Jim Zhu. 2015. “The Probability of Backtest Overfitting.” *Journal of
Computational Finance*. <https://doi.org/10.2139/ssrn.2326253>.

Bailey, David H., and Marcos López de Prado. 2014. “The Deflated Sharpe
Ratio: Correcting for Selection Bias, Backtest Overfitting and
Non-Normality.” *Journal of Portfolio Management* 40 (5): 94–107.
<https://doi.org/10.2139/ssrn.2460551>.